# Etapa 1: Importação das bibliotecas

In [ ]:
import torch.nn as nn
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.model_selection import train_test_split

np.random.seed(0)
torch.manual_seed(0)

## Etapa 2: Carregando e preparando base de dados

In [ ]:
with open('data.pkl', 'rb') as f:
    data = pickle.load(f)

with open('data_segmented.pkl', 'rb') as f:
    labels = pickle.load(f)

In [ ]:
type(data), type(labels)

(numpy.ndarray, numpy.ndarray)

In [ ]:
data.shape, labels.shape

((199, 256, 256, 3), (199, 256, 256))

In [ ]:
data_train, data_test, label_train, label_test = train_test_split(data, labels, test_size=0.2, random_state=42)

In [ ]:
data_train.shape, data_test.shape, label_train.shape, label_test.shape

((159, 256, 256, 3), (40, 256, 256, 3), (159, 256, 256), (40, 256, 256))

In [ ]:
data_train = torch.tensor(data_train, dtype=torch.float)
data_test = torch.tensor(data_test, dtype=torch.float)
label_train = torch.tensor(label_train, dtype=torch.long)
label_test = torch.tensor(label_test, dtype=torch.long)

train = torch.utils.data.TensorDataset(data_train, label_train)
test = torch.utils.data.TensorDataset(data_test, label_test)

train_loader = torch.utils.data.DataLoader(train, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test, batch_size=32, shuffle=False)

In [ ]:
train_loader.dataset.tensors[0].shape, train_loader.dataset.tensors[1].shape

(torch.Size([159, 256, 256, 3]), torch.Size([159, 256, 256]))

## Etapa 3: Definição do modelo

In [ ]:
class UNet(nn.Module):
    def __init__(self, input_channels, num_classes):
        super(UNet, self).__init__()
        
        # Encoder (Downsampling)
        self.enc1 = self.conv_block(input_channels, 32)
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2 = self.conv_block(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        
        self.enc3 = self.conv_block(64, 128)
        self.pool3 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = self.conv_block(128, 256)
        
        # Decoder (Upsampling)
        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec1 = self.conv_block(256, 128)
        
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = self.conv_block(128, 64)
        
        self.up3 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec3 = self.conv_block(64, 32)
        
        # Output layer
        self.out_conv = nn.Conv2d(32, num_classes, kernel_size=1)
    
    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder
        c1 = self.enc1(x)
        p1 = self.pool1(c1)
        
        c2 = self.enc2(p1)
        p2 = self.pool2(c2)
        
        c3 = self.enc3(p2)
        p3 = self.pool3(c3)
        
        # Bottleneck
        c4 = self.bottleneck(p3)
        
        # Decoder
        u1 = self.up1(c4)
        u1 = torch.cat([u1, c3], dim=1)
        c5 = self.dec1(u1)
        
        u2 = self.up2(c5)
        u2 = torch.cat([u2, c2], dim=1)
        c6 = self.dec2(u2)
        
        u3 = self.up3(c6)
        u3 = torch.cat([u3, c1], dim=1)
        c7 = self.dec3(u3)
        
        outputs = self.out_conv(c7)
        return outputs
        



In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

input_channels = 3
num_classes = 2

model = UNet(input_channels, num_classes).to(device)
device

device(type='cpu')

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Etapa 4: Treinamento do modelo

In [ ]:
def training_loop(epoch, model, train_loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        inputs, labels = data
        inputs = inputs.permute(0, 3, 1, 2)  # De (B, H, W, C) para (B, C, H, W)
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print('Epoch {} loss: {}'.format(epoch, running_loss / len(train_loader)))

In [ ]:
for epoch in range(10):
    model.train()
    print('Treinando ...')
    training_loop(epoch, model, train_loader, criterion, optimizer)

Treinando ...
